# Fusion trust scoring (Layer 1 × Layer 2)

Explore escalated vs quarantined alerts. Requires gold output from:

```bash
python -m pipelines quality
python -m pipelines detect
python -m pipelines fuse
```

See `models/fusion_spec.md` for the trust formula and severity penalties.
If Layer 2 has no ranked alerts yet (ensemble gated on `MIN_EVENT_ROWS`),
section 5 runs a synthetic fuse demo.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path("..").resolve()
GOLD_ROOT = ROOT / "data" / "gold"

from pipelines.config import SEVERITY_PENALTY
from pipelines.detection.build import read_event_alerts
from pipelines.fusion.build import read_trust_alerts
from pipelines.fusion.trust_score import fuse
from pipelines.quality.build import read_quality_incidents

fused = read_trust_alerts(GOLD_ROOT)
incidents = read_quality_incidents(GOLD_ROOT)
alerts = read_event_alerts(GOLD_ROOT)

print(f"Fused alerts: {len(fused)}")
print(f"Layer 2 alerts: {len(alerts)}")
print(f"Layer 1 incidents: {len(incidents)}")
fused.head()

## 1. Escalated vs quarantined

In [ ]:
if fused.empty:
    print("No fused alerts — run quality → detect → fuse (or use section 5 synthetic demo)")
else:
    counts = fused["status"].value_counts()
    display(counts.to_frame("count"))
    print(f"Mean trust_score: {fused['trust_score'].mean():.3f}")
    print(f"Mean alert_score: {fused['alert_score'].mean():.3f}")

    fig, ax = plt.subplots(figsize=(5, 4))
    counts.plot(kind="bar", ax=ax, color=["#2a9d8f", "#e76f51"][: len(counts)])
    ax.set_ylabel("Alerts")
    ax.set_title("Escalated vs quarantined")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 2. Trust score vs alert score

In [ ]:
if not fused.empty:
    fig, ax = plt.subplots(figsize=(7, 5))
    for status, group in fused.groupby("status"):
        ax.scatter(
            group["alert_score"],
            group["trust_score"],
            alpha=0.75,
            label=status,
        )
    ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1, label="trust = alert")
    ax.set_xlabel("alert_score (Layer 2)")
    ax.set_ylabel("trust_score (fusion)")
    ax.set_title("Severity penalty pulls quarantined points below the diagonal")
    ax.legend()
    ax.set_xlim(0, 1.05)
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.show()
else:
    print("Skip — no fused gold yet.")

## 3. Quarantine reasons (severity and rules)

In [ ]:
if fused.empty:
    print("Skip — no fused gold yet.")
else:
    quarantined = fused[fused["status"] == "quarantined"]
    if quarantined.empty:
        print("No quarantined alerts in gold.")
    else:
        display(
            quarantined[
                [
                    "locationid",
                    "date_local",
                    "parameter",
                    "alert_score",
                    "trust_score",
                    "max_severity",
                    "incident_count",
                    "incident_rule_ids",
                ]
            ].sort_values("trust_score")
        )
        print("Severity penalties:", SEVERITY_PENALTY)
        display(quarantined["max_severity"].value_counts().to_frame("quarantined_alerts"))

## 4. Layer 1 incident coverage on alert days

In [ ]:
if fused.empty:
    print("Skip — no fused gold yet.")
else:
    overlap = (
        fused.groupby(["locationid", "date_local"], as_index=False)
        .agg(
            n_alerts=("status", "size"),
            has_quality_incident=("has_quality_incident", "max"),
            max_severity=("max_severity", "first"),
        )
    )
    display(overlap)
    rate = float(overlap["has_quality_incident"].mean())
    print(f"Share of alert station-days with a Layer 1 incident: {rate:.1%}")

## 5. Synthetic fuse demo (when gold alerts are empty)

Mirrors fixtures in `tests/test_fusion.py`: one clean day (escalate) and one
stuck-sensor day (quarantine with high-severity penalty).

In [ ]:
from pipelines.detection.ensemble import EVENT_ALERT_COLUMNS
from pipelines.quality.rules import INCIDENT_COLUMNS

demo_alerts = pd.DataFrame(
    [
        {
            "locationid": 1544061,
            "date_local": "2026-01-08",
            "parameter": "pm25",
            "region_id": "sydney_metro",
            "alert_score": 0.8,
            "rank": 1,
            "if_flag": True,
            "lof_flag": True,
            "dbscan_flag": False,
            "agreement_count": 2,
            "weak_label": False,
            "feature_snapshot": "{}",
        },
        {
            "locationid": 1601414,
            "date_local": "2026-01-08",
            "parameter": "pm25",
            "region_id": "sydney_metro",
            "alert_score": 0.6,
            "rank": 2,
            "if_flag": True,
            "lof_flag": False,
            "dbscan_flag": False,
            "agreement_count": 1,
            "weak_label": False,
            "feature_snapshot": "{}",
        },
    ],
    columns=EVENT_ALERT_COLUMNS,
)

demo_incidents = pd.DataFrame(
    [
        {
            "locationid": 1544061,
            "date_local": "2026-01-08",
            "rule_id": "R2",
            "incident_type": "stuck_sensor",
            "severity": "high",
            "event_code": "E3",
            "metric_snapshot": "{}",
            "is_incident": True,
            "source": "rule",
        }
    ],
    columns=INCIDENT_COLUMNS,
)

demo = fuse(demo_incidents, demo_alerts)
display(
    demo[
        [
            "locationid",
            "date_local",
            "alert_score",
            "trust_score",
            "status",
            "max_severity",
            "incident_rule_ids",
        ]
    ]
)

expected_quarantine = 0.8 * (1 - SEVERITY_PENALTY["high"])
print(f"Expected quarantined trust_score: {expected_quarantine:.2f} (0.8 × 0.3)")
print("Clean station (1601414) should escalate with trust_score == alert_score.")

## Takeaways

- Coincident Layer 1 incidents → **quarantined** (still visible, never deleted)
- Clean station-days → **escalated** with full `alert_score` as `trust_score`
- High severity (e.g. R2 stuck sensor) applies a 0.7 penalty
- Unlock real fused gold by ingesting 30+ days so Layer 2 produces alerts